# Sequential BFS Baseline (for comparison with the parallel version)

In [1]:
class Graph:
    def __init__(self):
        self.adj = {}

    def add_node(self, node):
        if node not in self.adj:
            self.adj[node] = []

    def add_edge(self, u, v, weight=1):
        self.add_node(u)
        self.add_node(v)
        self.adj[u].append(v)
        self.adj[v].append(u)  # undirected for this BFS use case

    def nodes(self):
        return list(self.adj.keys())

    def neighbors(self, u):
        return self.adj.get(u, [])

    def num_nodes(self):
        return len(self.adj)

    def num_edges(self):
        return sum(len(v) for v in self.adj.values()) // 2

## Sequential BFS Implementation

In [3]:
from collections import deque

def sequential_bfs(graph, source):
    """
    Standard single-threaded BFS. Returns (visited_order, distances).
    """
    visited = {source}
    distances = {source: 0}
    order = [source]
    queue = deque([source])

    while queue:
        u = queue.popleft()
        for v in graph.neighbors(u):
            if v not in visited:
                visited.add(v)
                distances[v] = distances[u] + 1
                order.append(v)
                queue.append(v)

    return order, distances

## Sanity Test

In [4]:
g = Graph()
edges = [
    ("A", "B"), ("A", "C"), ("B", "D"), ("C", "D"),
    ("D", "E"), ("E", "F"), ("C", "F"),
]
for u, v in edges:
    g.add_edge(u, v)

order, distances = sequential_bfs(g, "A")
print("Visit order:", order)
print("Distances from A:", distances)

Visit order: ['A', 'B', 'C', 'D', 'F', 'E']
Distances from A: {'A': 0, 'B': 1, 'C': 1, 'D': 2, 'F': 2, 'E': 3}


## Generate a Large Benchmark Graph

In [5]:
import random

def generate_random_graph(n_nodes, avg_degree=6, seed=42):
    rng = random.Random(seed)
    g = Graph()
    node_names = list(range(n_nodes))
    for name in node_names:
        g.add_node(name)

    for u in node_names:
        targets = rng.sample(node_names, min(avg_degree, n_nodes - 1))
        for v in targets:
            if v != u:
                g.add_edge(u, v)

    return g

big_graph = generate_random_graph(n_nodes=20000, avg_degree=6)
print(f"Nodes: {big_graph.num_nodes()}, Edges: {big_graph.num_edges()}")

Nodes: 20000, Edges: 119996


## Time the Sequential BFS Baseline

In [6]:
import time

t0 = time.perf_counter()
order, distances = sequential_bfs(big_graph, 0)
sequential_time = time.perf_counter() - t0

print(f"Nodes visited: {len(order)} / {big_graph.num_nodes()}")
print(f"Sequential BFS time: {sequential_time*1000:.2f} ms")

Nodes visited: 20000 / 20000
Sequential BFS time: 30.86 ms
